In [ ]:
!pip install vaderSentiment -q

import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack, csr_matrix
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

analyzer = SentimentIntensityAnalyzer()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)

In [ ]:
import pandas as pd

df = pd.read_csv('/content/final_combined_enriched_v4.csv')
print(f"Total tweets : {len(df):,}")
print(f"Columns      : {df.columns.tolist()}")
df.head(3)

In [ ]:
df = df.drop(columns=['type'])

if 'user' not in df.columns:
    df['user'] = ['user_' + str(i) for i in range(len(df))]

df_unlabeled = df[['user', 'tweet']].copy()

df_unlabeled = df_unlabeled.drop_duplicates(subset='tweet').reset_index(drop=True)
df_unlabeled = df_unlabeled[df_unlabeled['tweet'].str.strip() != ''].reset_index(drop=True)

print("Unlabeled shape :", df_unlabeled.shape)
print(df_unlabeled.head())


In [ ]:

print(df.shape)


In [ ]:
df.isnull().sum()
df.duplicated().sum()

In [ ]:
import re

df['mention'] = df['tweet'].astype(str).apply(
    lambda x: re.findall(r'@(\w+)', x)
)

In [ ]:
df[['tweet','mention']].head(10)


In [ ]:
from collections import Counter

tweet_counts = Counter()
for mentions in df['mention']:
    for name in set(mentions):
        tweet_counts[name] += 1

top_200_tweets = tweet_counts.most_common(200)

top_200_tweets_df = pd.DataFrame(top_200_tweets, columns=['mentions', 'tweet_count'])

total_tweets = len(df)
top_200_tweets_df['percentage'] = (top_200_tweets_df['tweet_count'] / total_tweets * 100).round(2)

print(top_200_tweets_df)

In [ ]:
import re

no_mentions_df = df[df['mention'].apply(len) == 0]

no_mentions_df = no_mentions_df.copy()
no_mentions_df['hashtags'] = no_mentions_df['tweet'].apply(lambda x: re.findall(r'#(\w+)', x))

total_no_mention_tweets = len(no_mentions_df)
tweets_with_hashtags = (no_mentions_df['hashtags'].apply(len) > 0).sum()
pct_with_hashtags = round(tweets_with_hashtags / total_no_mention_tweets * 100, 2)

print(f"Tweets with no mentions: {total_no_mention_tweets}")
print(f"Of those, tweets with at least one hashtag: {tweets_with_hashtags} ({pct_with_hashtags}%)")

total_hashtag_occurrences = sum(len(h) for h in no_mentions_df['hashtags'])
print(f"Total hashtag occurrences in no-mention tweets: {total_hashtag_occurrences}")

In [ ]:
from collections import Counter
import re
df['hashtags'] = df['tweet'].astype(str).apply(lambda x: re.findall(r'#(\w+)', x.lower()))
hashtag_counts = Counter(tag for tags in df['hashtags'] for tag in set(tags))

top_200_hashtags = hashtag_counts.most_common(200)
top_200_hashtags_df = pd.DataFrame(top_200_hashtags, columns=['hashtag', 'tweet_count'])

total_tweets = len(df)
top_200_hashtags_df['percentage'] = (top_200_hashtags_df['tweet_count'] / total_tweets * 100).round(2)

pd.set_option('display.max_rows', 200)
print(top_200_hashtags_df)

In [ ]:
non_political = set("'arbitrage', 'babarazam', 'bcci', 'bigbrother', 'bitcoin', 'bollywood', 'btc', 'btcinr', 'caatsa', 'chennai', 'consultants', 'covid', 'covid19', 'cricket', 'crickettwitter', 'crypto', 'dalailama', 'development', 'england', 'engvind', 'engvsind', 'food', 'health', 'indiancricketteam', 'indianews', 'indvseng', 'instagramreels', 'latestnews', 'license', 'love', 'maps', 'mapsofindia', 'media', 'monkeypox', 'monkeypoxvirus', 'news18', 'newsupdate', 'odi', 'reels', 'registration', 'rohitsharma', 'sanjusamson', 'sports', 'teamindia', 'tiktok', 'travel', 'twitter', 'viral', 'viratkohli𓃵', 'westindies', 'world', 'worldyouthskillsday', 'म',travel reels love mapsofindia maps crypto म chennai instagramreels caatsa cricket england kingkohli kohli भ latestreels viratkohli𓃵  indveng  coronavirusviratkohli indvseng engvsind engvind crickettwitter odi rohitsharma teamindia bcci indiancricketteam babarazam sports sanjusamson westindies worldcup bitcoin btc btcinr crypto arbitrage monkeypox monkeypoxvirus covid19 covid health bollywood tiktok reels instagramreels bigbrother viral love travel food twitter world maps mapsofindia worldyouthskillsday news18 indianews latestnews media consultants development registration license newsupdate dalailama".split())

political_hashtags_df = top_200_hashtags_df[~top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

removed_df = top_200_hashtags_df[top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

print(f"Kept {len(political_hashtags_df)} political hashtags | Removed {len(removed_df)} non-political")
print(sorted(removed_df['hashtag'].tolist()))


In [ ]:
political_counts = [
    (tag, count)
    for tag, count in hashtag_counts.most_common()
    if tag not in non_political
][:200]

top_200_after_removal_df = pd.DataFrame(
    political_counts, columns=['hashtag', 'tweet_count']
)
top_200_after_removal_df['percentage'] = (
    top_200_after_removal_df['tweet_count'] / total_tweets * 100
)

print(f"Top {len(top_200_after_removal_df)} political hashtags after removal")
top_200_after_removal_df

In [ ]:
print(df.columns.tolist())


In [ ]:
df['has_mention'] = df['mention'].apply(len) > 0
df['has_hashtag'] = df['hashtags'].apply(len) > 0

at_least_one = df[df['has_mention'] | df['has_hashtag']]

print(f"Tweets with at least one (mention or hashtag): {len(at_least_one)} "
      f"({round(len(at_least_one)/len(df)*100, 2)}%)")

at_least_one[['tweet', 'mention', 'hashtags']]

In [ ]:
STOPWORDS = set(stopwords.words('english'))

HINDI_SW = {
    'hai','hain','bhi','ka','ki','ke','ko','se','aur','main','mein',
    'nahi','aap','ho','toh','ye','yeh','wo','woh','ne','pe','kya',
    'tha','thi','kar','liye','phir','ab','ek','do','teen'
}

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split()
              if w not in STOPWORDS and w not in HINDI_SW and len(w) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['tweet'].apply(preprocess)
df = df.drop_duplicates(subset=['tweet']).reset_index(drop=True)
df = df[df['clean_text'].str.split().str.len() >= 3].reset_index(drop=True)

print(f"After preprocessing : {len(df):,} tweets")
print(f"\nBefore: {df['tweet'].iloc[2][:200]}")
print(f"After : {df['clean_text'].iloc[2][:200]}")

In [ ]:
!pip install textblob --break-system-packages
!python -m textblob.download_corpora


In [ ]:
tfidf   = TfidfVectorizer(max_features=5000, ngram_range=(1,1),
                           min_df=3, max_df=0.90, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df['clean_text'])

vocab       = tfidf.get_feature_names_out()
mean_scores = np.asarray(X_tfidf.mean(axis=0)).flatten()
doc_freq    = np.asarray((X_tfidf > 0).sum(axis=0)).flatten()
total_docs  = X_tfidf.shape[0]

top_1000_idx    = mean_scores.argsort()[-1000:][::-1]
sample_words  = [vocab[i] for i in top_1000_idx]
top_1000_scores = [float(mean_scores[i]) for i in top_1000_idx]
top_1000_dpct   = [float(doc_freq[i]/total_docs*100) for i in top_1000_idx]

print(f"TF-IDF full vocabulary : {len(vocab)}")
print(f"Top 1000 extracted     : 1,000")
for i in range(50):
    print(f"  {sample_words[i]:<20} score={top_1000_scores[i]:.5f}  doc%={top_1000_dpct[i]:.1f}%")


In [ ]:

NLTK_STOPWORDS = set(stopwords.words('english'))

DROP_FROM_VOCAB = {
    'good','bad','great','best','worst','nice','better','well',
    'really','very','much','many','every','even','just','like',
    'also','still','back','right','left','now','only','would',
    'could','should','want','think','know','say','said','see',
    'get','got','make','made','give','take','come','going',
    'amp','people','like','dont','one','know','team',
    'good','even','time','country','see','get','new','also',
    'never','think','would','man','much','every','many','world',

    # ── generic people/time/place words ───────────────────────
    'man','men','woman','women','people','person','time','day',
    'year','week','month','today','yesterday','tomorrow','way',
    'thing','things','place','world','country','city','state',
    'home','house','life','hand','eye','face','name','number',

    # ── common verbs (no political meaning) ───────────────────
    'said','told','asked','went','came','put','let','run','ran',
    'used','using','done','doing','look','looking','looked',
    'work','worked','working','help','helped','helping','try',
    'tried','trying','seem','seems','seemed','feel','felt',
    'show','showed','shown','keep','kept','call','called',
    'talk','talked','talking','write','wrote','written','read',
        'follow','followed','share','shared','sharing','post','posted',

    # ── twitter/social media artifacts ────────────────────────
    'amp','via','co','http','https','rt','please','need',
    'follow','retweet','tweet','tweeted','thread','account',
    'handle','profile','click','link','watch','read','check',

    # ── generic quantity/frequency words ──────────────────────
    'never','always','ever','already','yet','still','else',
    'more','less','most','least','few','many','lot','lots',
    'enough','enough','whole','full','half','part','bit',
    'some','any','all','both','each','every','either',
    'another','other','others','same','different','last','next',

    # ── generic adjectives (appear everywhere) ────────────────
    'new','old','big','small','large','long','short','high',
    'low','early','late','first','second','last','important',
    'real','true','false','wrong','right','strong','weak',
    'hard','easy','free','open','close','clear','common',
    'public','private','official','special','major','minor',

    # ── generic news/reporting language ───────────────────────
    'news','report','reported','reporting','says','claim',
    'claimed','statement','source','sources','according',
    'based','given','noted','added','confirmed','denied',
    'announced','announced','mentioned','revealed','stated',

    # ── common connector/filler words ─────────────────────────
    'also','however','therefore','thus','hence','though',
    'although','because','since','while','when','where',
    'which','whose','whom','whether','instead','despite',
    'against','towards','within','without','across','through',
    'during','before','after','above','below','between',
    'among','around','behind','beside','beyond','inside',

    # ── numbers and generic quantifiers ───────────────────────
    'one','two','three','four','five','six','seven','eight',
    'nine','ten','hundred','thousand','million','billion',
    'crore','lakh',

    # ── common Indian-English generic words ───────────────────
    'ji','sir','dear','respected','brother','sister','friend',
    'bhai','didi','agar','phir','bas','karo','karke','abhi',
    'wala','wali','wale','please','kindly','request','thanks',
    'thank','welcome','congrats','sorry','okay','yes','yeah',
    'nope','haha','lol','omg','wow','oh','ah','hmm',

    # ── generic action/state words ────────────────────────────
    'start','started','starting','stop','stopped','end',
    'ended','begin','began','continue','continued','change',
    'changed','move','moved','stand','stood','stay','stayed',
    'live','lived',

    # ── very common Indian political dataset noise ────────────
    # ── generic verbs ────────────────────────────────────────
    'need', 'great', 'years', 'well', 'want', 'say', 'come',
    'make', 'best', 'take', 'cant', 'understand', 'become',
    'give', 'made', 'play', 'work', 'keep', 'real', 'using',
    'used', 'said', 'since', 'bad', 'without', 'seen',
    'getting', 'let', 'got', 'yes', 'done', 'read', 'talk',
    'called', 'look', 'show', 'find', 'ask', 'put', 'run',
    'happen', 'call', 'shows', 'leave', 'create', 'stay',
    'start', 'bring', 'try', 'follow', 'speak', 'tell',

    # ── generic adjectives ───────────────────────────────────
    'thats', 'better', 'may', 'big', 'doesnt', 'really',
    'real', 'worst', 'happy', 'long', 'dirty', 'true',
    'hard', 'strong', 'greatest', 'ultimate', 'unbeatable',
    'interesting', 'full', 'low', 'common', 'entire', 'huge',
    'completely', 'clear', 'young', 'sure', 'different',
    'wrong', 'high', 'least', 'possible', 'next', 'first',
    'always', 'nothing', 'ever', 'please', 'today', 'sir',

    # ── generic nouns ────────────────────────────────────────
    'person', 'day', 'thing', 'family', 'name', 'life',
    'level', 'game', 'things', 'two', 'etc', 'way', 'year',
    'times', 'moment', 'sense', 'idea', 'point', 'chance',
    'words', 'side', 'place', 'back', 'fact', 'days', 'lot',
    'part', 'birthday', 'book', 'story', 'job', 'example',
    'difference', 'birth', 'home', 'mind', 'thought',

    # ── generic connectors/fillers ────────────────────────────
    'going', 'still', 'away', 'making', 'yet', 'must',
    'already', 'based', 'towards', 'around', 'coming',
    'though', 'never', 'else', 'rather', 'beyond', 'past',
    'last', 'behind', 'general', 'whatever', 'single',
    'others', 'cannot', 'among', 'without', 'instead',
    'definitely', 'completely', 'especially', 'following',
    'actually', 'someone', 'anyone', 'everyone', 'something',
    'everything', 'anything', 'nothing', 'enough', 'till',

    # ── twitter noise ────────────────────────────────────────
    'twitter', 'tweet', 'via', 'lol', 'ppl', 'hes', 'youre',
    'isnt', 'didnt', 'wont', 'thats', 'whats', 'dont',
    'cant', 'doesnt', 'lets', 'youll', 'its',

    # ── sports/entertainment (non-political) ─────────────────
    'cricket', 'bcci', 'sports', 'team', 'player', 'players',
    'playing', 'played', 'film', 'movie', 'bollywood', 'cinema',
    'fans', 'star', 'sanju', 'game', 'play',

    # ── generic sentiments (no political specificity) ─────────
    'happy', 'sad', 'love', 'hope', 'wish', 'wishes',
    'sorry', 'welcome', 'thanks', 'congratulations', 'dear',
    'care', 'feel', 'feeling', 'respect', 'proud',

    # ── common people references ─────────────────────────────
    'man', 'men', 'woman', 'women', 'guy', 'guys', 'lady',
    'king', 'hero', 'person', 'human',

    # ── generic misc ─────────────────────────────────────────
    'matter', 'matters', 'remain', 'seems', 'relevant',
    'irrelevant', 'understanding', 'become', 'use', 'self',
    'origin', 'role', 'black', 'white', 'old', 'post',
    'watch', 'watching', 'global', 'learn', 'reason', 'agree',
    'born', 'means', 'makes', 'join', 'forget', 'comment',
    'involved', 'start', 'face', 'top', 'position',
}




# combine both into one set
ALL_STOP = NLTK_STOPWORDS | DROP_FROM_VOCAB

MIN_WORD_LEN = 3
MAX_DOC_PCT  = 60.0

political_vocab = []
removed_vocab   = []

for word, score, dpct in zip(sample_words, top_1000_scores, top_1000_dpct):
    if word.lower() in ALL_STOP:
        removed_vocab.append((word, score, dpct, 'stopword/no political meaning')); continue
    if len(word) < MIN_WORD_LEN:
        removed_vocab.append((word, score, dpct, 'too short')); continue
    if dpct > MAX_DOC_PCT:
        removed_vocab.append((word, score, dpct, f'in {dpct:.0f}% docs')); continue
    political_vocab.append((word, score, dpct))







In [ ]:
print(f"After filtering  : {len(political_vocab):,} words")
print(f"Removed          : {len(DROP_FROM_VOCAB ):,} words")
print()
for i, (w, s, d) in enumerate(political_vocab[:50]):
    print(f"  {i+1:<4} {w:<25} score={s:.5f}  doc%={d:.1f}%")

In [ ]:
clean_vocab_list = [w for w, s, d in political_vocab[:700]]

tfidf_pol = TfidfVectorizer(
    vocabulary    = clean_vocab_list,  # restrict to cleaned vocab only
    sublinear_tf  = True
)

X_pol_only    = tfidf_pol.fit_transform(df['clean_text'])
political_cols = list(range(X_pol_only.shape[1]))   # all cols = political cols

political_score = np.asarray(X_pol_only.sum(axis=1)).flatten()
df['political_score'] = political_score

non_zero  = political_score[political_score > 0]
threshold = float(np.median(non_zero))
df['is_political'] = (df['political_score'] > 0).astype(int)

pol_count    = int(df['is_political'].sum())
nonpol_count = int((df['is_political']==0).sum())

print(f"Vocabulary used              : {len(clean_vocab_list)}")
print(f"Identified as POLITICAL      : {pol_count:,}  ({pol_count/len(df)*100:.1f}%)")
print(f"Identified as NON-POLITICAL  : {nonpol_count:,}  ({nonpol_count/len(df)*100:.1f}%)")
print()
print("top  words used:")
for w in clean_vocab_list[:1000]:
    print(f"  {w}")

print()


In [ ]:

from collections import Counter as _Counter
_mention_counter = _Counter(m for ms in df['mention'] for m in ms)
_hashtag_counter = _Counter(h for hs in df['hashtags'] for h in hs)
political_mentions = {m for m, _ in _mention_counter.most_common(200)}
political_hashtags = {h for h, _ in _hashtag_counter.most_common(200)}
political_mentions = {m.lower() for m in political_mentions}
political_hashtags = {h.lower() for h in political_hashtags}

MENTION_WEIGHT = 1.0
HASHTAG_WEIGHT = 1.0
CAP = 3

def political_mention_hashtag_score(mentions, hashtags):
    mention_hits = sum(1 for m in mentions if m.lower() in political_mentions)
    hashtag_hits = sum(1 for h in hashtags if h.lower() in political_hashtags)

    capped_mention_hits = min(mention_hits, CAP)
    capped_hashtag_hits = min(hashtag_hits, CAP)

    score = (capped_mention_hits * MENTION_WEIGHT) + (capped_hashtag_hits * HASHTAG_WEIGHT)
    has_signal = int((mention_hits > 0) or (hashtag_hits > 0))

    return pd.Series({
        'mention': mention_hits,
        'hashtag': hashtag_hits,
        'pol_score': score,
        'has_political_signal_mh': has_signal
    })

df[['political_mention_hits', 'political_hashtag_hits',
    'political_score_mh', 'has_political_signal_mh']] = df.apply(
    lambda row: political_mention_hashtag_score(row['mention'], row['hashtags']), axis=1
)

print("Both mention+hashtag hit:", len(df[(df['political_mention_hits'] > 0) & (df['political_hashtag_hits'] > 0)]))
print("Only mention hit:        ", len(df[(df['political_mention_hits'] > 0) & (df['political_hashtag_hits'] == 0)]))
print("Only hashtag hit:        ", len(df[(df['political_mention_hits'] == 0) & (df['political_hashtag_hits'] > 0)]))
print("Neither:                 ", len(df[(df['political_mention_hits'] == 0) & (df['political_hashtag_hits'] == 0)]))

df[['tweet', 'political_mention_hits', 'political_hashtag_hits',
    'political_score_mh', 'has_political_signal_mh']].head(10)

In [ ]:
import re
import pandas as pd

# ============================================================
# STEP 1 — Verify mention and hashtag extraction
# ============================================================

if 'mention' not in df.columns:
    df['mention'] = df['tweet'].astype(str).apply(lambda x: [m.lower() for m in re.findall(r'@(\w+)', x)])

if 'hashtags' not in df.columns:
    df['hashtags'] = df['tweet'].astype(str).apply(lambda x: [h.lower() for h in re.findall(r'#(\w+)', x)])

df['has_mention'] = df['mention'].apply(len) > 0
df['has_hashtag'] = df['hashtags'].apply(len) > 0
at_least_one = df[df['has_mention'] | df['has_hashtag']]

total_tweets = len(df)
n_with_mention = df['has_mention'].sum()
n_with_hashtag = df['has_hashtag'].sum()
n_with_either = len(at_least_one)

print("STEP 1 — Extraction verification")
print(f"Total tweets                 : {total_tweets:,}")
print(f"Tweets with mentions         : {n_with_mention:,} ({n_with_mention/total_tweets*100:.2f}%)")
print(f"Tweets with hashtags         : {n_with_hashtag:,} ({n_with_hashtag/total_tweets*100:.2f}%)")
print(f"Tweets with either           : {n_with_either:,} ({n_with_either/total_tweets*100:.2f}%)")

df[['tweet', 'mention', 'hashtags']].head(5)


# ============================================================
# STEP 2 — Mention frequency table
# ============================================================

mention_freq = df['mention'].apply(set).explode().value_counts()
mention_freq_df = mention_freq.reset_index()
mention_freq_df.columns = ['mention', 'frequency']
mention_freq_df = mention_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 2 — Mention frequency table (top 20)")
mention_freq_df.head(20)


# ============================================================
# STEP 3 — Hashtag frequency table
# ============================================================

hashtag_freq = df['hashtags'].apply(set).explode().value_counts()
hashtag_freq_df = hashtag_freq.reset_index()
hashtag_freq_df.columns = ['hashtag', 'frequency']
hashtag_freq_df = hashtag_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 3 — Hashtag frequency table (top 20)")
hashtag_freq_df.head(20)


# ============================================================
# STEP 4 — Frequency analysis / coverage
# ============================================================

total_unique_mentions = mention_freq_df['mention'].nunique()
total_unique_hashtags = hashtag_freq_df['hashtag'].nunique()
total_mention_occurrences = mention_freq_df['frequency'].sum()
total_hashtag_occurrences = hashtag_freq_df['frequency'].sum()

print("\nSTEP 4 — Frequency analysis")
print(f"Total unique mentions     : {total_unique_mentions:,}")
print(f"Total unique hashtags     : {total_unique_hashtags:,}")
print(f"Total mention occurrences : {total_mention_occurrences:,}")
print(f"Total hashtag occurrences : {total_hashtag_occurrences:,}")

# coverage of dataset by top-N most frequent entities
for n in [50, 100, 200, 500]:
    top_n_mentions = set(mention_freq_df['mention'].head(n))
    top_n_hashtags = set(hashtag_freq_df['hashtag'].head(n))

    cov_mention = df['mention'].apply(lambda names: bool(set(names) & top_n_mentions)).sum()
    cov_hashtag = df['hashtags'].apply(lambda tags: bool(set(tags) & top_n_hashtags)).sum()

    print(f"\nTop {n} mentions cover : {cov_mention:,} tweets ({cov_mention/total_tweets*100:.2f}%)")
    print(f"Top {n} hashtags cover : {cov_hashtag:,} tweets ({cov_hashtag/total_tweets*100:.2f}%)")


# ============================================================
# STEP 5 — Prepare clean tables for manual labeling
# ============================================================

political_mention_labeling_table = mention_freq_df.copy()
political_mention_labeling_table['Label'] = ""

political_hashtag_labeling_table = hashtag_freq_df.copy()
political_hashtag_labeling_table['Label'] = ""

print("\nSTEP 5 — Labeling tables ready")
political_mention_labeling_table.head(20)
political_hashtag_labeling_table.head(20)

# export for manual labeling
political_mention_labeling_table.to_csv('political_mention_labeling_table.csv', index=False)
political_hashtag_labeling_table.to_csv('political_hashtag_labeling_table.csv', index=False)

In [ ]:
import re
import pandas as pd
from IPython.display import display

# ============================================================
# STEP 1 — Verify mention and hashtag extraction
# ============================================================

if 'mention' not in df.columns:
    df['mention'] = df['tweet'].astype(str).apply(lambda x: [m.lower() for m in re.findall(r'@(\w+)', x)])

if 'hashtags' not in df.columns:
    df['hashtags'] = df['tweet'].astype(str).apply(lambda x: [h.lower() for h in re.findall(r'#(\w+)', x)])

df['has_mention'] = df['mention'].apply(len) > 0
df['has_hashtag'] = df['hashtags'].apply(len) > 0
at_least_one = df[df['has_mention'] | df['has_hashtag']]

print(f"Tweets with at least one (mention or hashtag): {len(at_least_one)} "
      f"({round(len(at_least_one)/len(df)*100, 2)}%)")
at_least_one[['tweet', 'mention', 'hashtags']].to_csv('tweets_with_mention_or_hashtag.csv', index=False)
print(f"Saved {len(at_least_one):,} tweets to tweets_with_mention_or_hashtag.csv")

# ============================================================
# STEP 2 — Mention frequency table (ALL unique mentions)
# ============================================================

mention_freq = df['mention'].apply(set).explode().value_counts()
mention_freq_df = mention_freq.reset_index()
mention_freq_df.columns = ['mention', 'frequency']
mention_freq_df = mention_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 2 — Mention frequency table ", len(mention_freq_df), "unique mentions)")
display(mention_freq_df.head(200))


# ============================================================
# STEP 3 — Hashtag frequency table (ALL unique hashtags)
# ============================================================

hashtag_freq = df['hashtags'].apply(set).explode().value_counts()
hashtag_freq_df = hashtag_freq.reset_index()
hashtag_freq_df.columns = ['hashtag', 'frequency']
hashtag_freq_df = hashtag_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 3 — Hashtag frequency table (top 20 of", len(hashtag_freq_df), "unique hashtags)")
display(hashtag_freq_df.head(200))






In [ ]:
top_200_mentions = set(mention_freq_df.head(200)['mention'])
top_200_hashtags = set(hashtag_freq_df.head(200)['hashtag'])

df['top_200_mention'] = df['mention'].apply(lambda names: bool(set(names) & top_200_mentions))
df['top_200_hashtag'] = df['hashtags'].apply(lambda tags: bool(set(tags) & top_200_hashtags))

mention_only = (df['top_200_mention'] & ~df['top_200_hashtag']).sum()
hashtag_only = (~df['top_200_mention'] & df['top_200_hashtag']).sum()
both = (df['has_top200_mention'] & df['has_top200_hashtag']).sum()
either = (df['has_top200_mention'] | df['has_top200_hashtag']).sum()

total_tweets = len(df)

print(f"Total tweets                                  : {total_tweets:,}")
print(f"Top-200 mention only                          : {mention_only:,} ({mention_only/total_tweets*100:.2f}%)")
print(f"Top-200 hashtag only                          : {hashtag_only:,} ({hashtag_only/total_tweets*100:.2f}%)")
print(f"Both top-200 mention and hashtag               : {both:,} ({both/total_tweets*100:.2f}%)")
print(f"Either (union, at least one top-200 signal)    : {either:,} ({either/total_tweets*100:.2f}%)")

In [ ]:
non_political = set("""
bcci imvkohli youtube imro45 akshaykumar twitter elonmusk iamsanjusamson
""".split())

political_mentions_df = top_200_mentions_df[~top_200_mentions_df['mention'].isin(non_political)].reset_index(drop=True)

removed_df = top_200_mentions_df[top_200_mentions_df['mention'].isin(non_political)].reset_index(drop=True)

print(f"Kept {len(political_mentions_df)} political mentions | Removed {len(removed_df)} non-political")
print(sorted(removed_df['mention'].tolist()))
non_political = set("""
arbitrage babarazam bcci bigbrother bitcoin bollywood btc btcinr caatsa chennai
consultants covid covid19 cricket crickettwitter crypto dalailama development
england engvind engvsind food health indiancricketteam indianews indvseng
instagramreels latestnews license love maps mapsofindia media monkeypox
monkeypoxvirus news18 newsupdate odi reels registration rohitsharma sanjusamson
sports teamindia tiktok travel twitter viral viratkohli𓃵 westindies world
worldyouthskillsday म chennai kingkohli kohli भ latestreels indveng
coronavirusviratkohli reelsindia tiktoks worldcup
""".split())

political_hashtags_df = top_200_hashtags_df[~top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

removed_df = top_200_hashtags_df[top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

print(f"Kept {len(political_hashtags_df)} political hashtags | Removed {len(removed_df)} non-political")
print(sorted(removed_df['hashtag'].tolist()))

In [ ]:
mention_freq = df['mention'].apply(set).explode().value_counts()
mention_freq_df = mention_freq.reset_index()
mention_freq_df.columns = ['mention', 'frequency']
mention_freq_df = mention_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 2 — Mention frequency table ", len(mention_freq_df), "unique mentions)")
display(mention_freq_df.head(300))

In [ ]:
non_political = set("""
arbitrage babarazam bcci bigbrother bitcoin bollywood btc btcinr caatsa chennai
consultants covid covid19 cricket crickettwitter crypto dalailama development
england engvind engvsind food health indiancricketteam indianews indvseng
instagramreels latestnews license love maps mapsofindia media monkeypox
monkeypoxvirus news18 newsupdate odi reels registration rohitsharma sanjusamson
sports teamindia tiktok travel twitter viral viratkohli𓃵 westindies world
worldyouthskillsday म chennai kingkohli kohli भ latestreels indveng
coronavirusviratkohli reelsindia tiktoks worldcup,peace,jaspritbumrah, ipl2019, kingkohli, kohli, maps, viratkohli𓃵
""".split())

political_hashtags_df = top_200_hashtags_df[~top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

removed_df = top_200_hashtags_df[top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

print(f"Kept {len(political_hashtags_df)} political hashtags | Removed {len(removed_df)} non-political")
print(sorted(removed_df['hashtag'].tolist()))

In [ ]:
import re

hashtag_freq = df['hashtags'].apply(set).explode().value_counts()
hashtag_freq_df = hashtag_freq.reset_index()
hashtag_freq_df.columns = ['hashtag', 'frequency']
hashtag_freq_df = hashtag_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 3 - Hashtag frequency table ", len(hashtag_freq_df), "unique hashtags)")
display(hashtag_freq_df.head(300))

# ──── non_political exclusion set, fixed to split on both commas and whitespace ────
non_political_hashtags_new = set(""" ukrainerussiawar russia usa europe nato china india africa quote freedom freespeech corona coronavirus boosterdose vaccines coronavaccines dainiikgomantak dainiikgomantaknews farmersprotest म ज घर_व babarazam viratkohli crickettwitter icc oscar msp_च greenenergy cleanenergy hydrogen fuelcell windpower solarenergy renewableenergy cleanseas renewables climatechange morocco climatecrisis plasticwaste pcn piracy bisleri monkeypox thiruvananthapuram uae """.split())

political_hashtags_df = hashtag_freq_df[~hashtag_freq_df['hashtag'].isin(non_political_hashtags_new)].reset_index(drop=True)

removed_df = hashtag_freq_df[hashtag_freq_df['hashtag'].isin(non_political_hashtags_new)].reset_index(drop=True)

print(f"Kept {len(political_hashtags_df)} political hashtags | Removed {len(removed_df)} non-political")
print(sorted(removed_df['hashtag'].tolist()))

In [ ]:
top_300_political_hashtags = set(political_hashtags_df.head(300)['hashtag'])

df['has_top300_political_hashtag'] = df['hashtags'].apply(lambda tags: bool(set(tags) & top_300_political_hashtags))

n_covered_300 = df['has_top300_political_hashtag'].sum()
total_tweets = len(df)

print(f"Total tweets                                : {total_tweets:,}")
print(f" top-300 political hashtag: {n_covered_300:,} ({n_covered_300/total_tweets*100:.2f}%)")

In [ ]:
top_200_political_hashtags = set(political_hashtags_df.head(200)['hashtag'])

df['has_top200_political_hashtag'] = df['hashtags'].apply(lambda tags: bool(set(tags) & top_200_political_hashtags))

n_covered = df['has_top200_political_hashtag'].sum()
total_tweets = len(df)

print(f"Total tweets                                : {total_tweets:,}")
print(f"Tweets containing a top-200 political hashtag: {n_covered:,} ({n_covered/total_tweets*100:.2f}%)")

In [ ]:
# ============================================================
# STEP 4 - Entity-level binary label dictionaries
# Political entities (top-200 mentions + top-200 hashtags) -> 1
# Non-political entities (removed via non_political filter) -> 0
# NOTE: entity-level labels only (not tweet labels)
# ============================================================
import pandas as pd
#
# 1) Build the MENTION frequency table (mirrors hashtag_freq_df)
mention_freq = df['mention'].apply(set).explode().value_counts()
mention_freq_df = mention_freq.reset_index()
mention_freq_df.columns = ['mention', 'frequency']
mention_freq_df = mention_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)
#
political_mentions_df = mention_freq_df[~mention_freq_df['mention'].isin(non_political)].reset_index(drop=True)
removed_mentions_df = mention_freq_df[mention_freq_df['mention'].isin(non_political)].reset_index(drop=True)
print(f"Kept {len(political_mentions_df)} political mentions | Removed {len(removed_mentions_df)} non-political")
print(sorted(removed_mentions_df['mention'].tolist()))
#
# 3) Curated entity lists (normalized: stripped + lowercased)
top_200_political_mentions = set(political_mentions_df.head(200)['mention'])
top_200_political_hashtags = set(political_hashtags_df.head(200)['hashtag'])
political_entities = sorted({str(e).strip().lower() for e in (top_200_political_mentions | top_200_political_hashtags) if str(e).strip()})
removed_words = set(removed_df['hashtag']) | set(removed_mentions_df['mention'])
non_political_words = sorted({str(w).strip().lower() for w in removed_words if str(w).strip()})
#
# 4) Build the binary lookup dictionaries (entity -> label)
political_dict = {e: 1 for e in political_entities}
non_political_dict = {w: 0 for w in non_political_words}
#
# 5) Report dictionary sizes
print('\n=== Dictionary sizes ===')
print('Political mentions (top-200) :', len(top_200_political_mentions))
print('Political hashtags (top-200) :', len(top_200_political_hashtags))
print('Political dictionary (label=1):', len(political_dict))
print('Non-political dictionary (0)  :', len(non_political_dict))
#
# 6) Verify there are no duplicate entries within each dictionary
pol_raw = [str(e).strip().lower() for e in (list(top_200_political_mentions) + list(top_200_political_hashtags)) if str(e).strip()]
nonpol_raw = [str(w).strip().lower() for w in removed_words if str(w).strip()]
pol_dupes = sorted({t for t in pol_raw if pol_raw.count(t) > 1})
nonpol_dupes = sorted({t for t in nonpol_raw if nonpol_raw.count(t) > 1})
print('\n=== Duplicate check (within each dictionary) ===')
print('Political duplicates     :', len(pol_dupes), pol_dupes if pol_dupes else '(none)')
print('Non-political duplicates :', len(nonpol_dupes), nonpol_dupes if nonpol_dupes else '(none)')
#
# 7) Check overlaps between political and non-political dicts (report only)
overlap = sorted(set(political_dict) & set(non_political_dict))
print('\n=== Overlap check (political vs non-political) ===')
print('Overlapping entities     :', len(overlap))
print('CONFLICTS:', overlap if overlap else 'None - dictionaries are disjoint.')
#
# 8) Build lookup tables (DataFrames) in Entity/Label format
political_label_df = pd.DataFrame({'Entity': list(political_dict.keys()), 'Label': list(political_dict.values())})
non_political_label_df = pd.DataFrame({'Entity': list(non_political_dict.keys()), 'Label': list(non_political_dict.values())})
print('\n=== Political dictionary (label=1) preview ===')
display(political_label_df.head(100))
print('\n=== Non-political dictionary (label=0) preview ===')
display(non_political_label_df.head(100))

In [ ]:
# ============================================================
# STEP 5 - Entity-level binary labels, HASHTAGS and MENTIONS SEPARATELY
#   political -> 1 (kept)   |   non-political -> 0 (removed via non_political)
#   entity-level labels only (not tweet labels)
# ============================================================
import pandas as pd
#
# --- HASHTAGS: top-200 political (1) + removed non-political (0) ---
top_200_political_hashtags = set(political_hashtags_df.head(200)['hashtag'])
removed_hashtag_words = set(removed_df['hashtag'])
pol_hashtags = sorted({str(h).strip().lower() for h in top_200_political_hashtags if str(h).strip()})
nonpol_hashtags = sorted({str(h).strip().lower() for h in removed_hashtag_words if str(h).strip()})
hashtag_political_dict = {h: 1 for h in pol_hashtags}
hashtag_non_political_dict = {h: 0 for h in nonpol_hashtags}
#
# --- MENTIONS: top-200 political (1) + removed non-political (0) ---
top_200_political_mentions = set(political_mentions_df.head(200)['mention'])
removed_mention_words = set(removed_mentions_df['mention'])
pol_mentions = sorted({str(m).strip().lower() for m in top_200_political_mentions if str(m).strip()})
nonpol_mentions = sorted({str(m).strip().lower() for m in removed_mention_words if str(m).strip()})
mention_political_dict = {m: 1 for m in pol_mentions}
mention_non_political_dict = {m: 0 for m in nonpol_mentions}
#
# --- Sizes ---
print('=== Dictionary sizes (separate) ===')
print('Hashtags political (1)   :', len(hashtag_political_dict))
print('Hashtags non-political(0):', len(hashtag_non_political_dict))
print('Mentions political (1)   :', len(mention_political_dict))
print('Mentions non-political(0):', len(mention_non_political_dict))
#
# --- Duplicate checks (within each dict) ---
h_pol_raw = [str(h).strip().lower() for h in political_hashtags_df.head(200)['hashtag'] if str(h).strip()]
m_pol_raw = [str(m).strip().lower() for m in political_mentions_df.head(200)['mention'] if str(m).strip()]
h_pol_dupes = sorted({t for t in h_pol_raw if h_pol_raw.count(t) > 1})
m_pol_dupes = sorted({t for t in m_pol_raw if m_pol_raw.count(t) > 1})
print('\n=== Duplicate check ===')
print('Hashtag political duplicates :', len(h_pol_dupes), h_pol_dupes if h_pol_dupes else '(none)')
print('Mention political duplicates :', len(m_pol_dupes), m_pol_dupes if m_pol_dupes else '(none)')
#
# --- Overlap checks (political vs non-political), per entity type, report only ---
hashtag_overlap = sorted(set(hashtag_political_dict) & set(hashtag_non_political_dict))
mention_overlap = sorted(set(mention_political_dict) & set(mention_non_political_dict))
print('\n=== Overlap check (political vs non-political) ===')
print('Hashtag overlaps :', len(hashtag_overlap), hashtag_overlap if hashtag_overlap else 'None - disjoint')
print('Mention overlaps :', len(mention_overlap), mention_overlap if mention_overlap else 'None - disjoint')
#
# --- Lookup tables (Entity/Label) per type ---
hashtag_label_df = pd.DataFrame([(h, 1) for h in pol_hashtags] + [(h, 0) for h in nonpol_hashtags], columns=['Entity', 'Label'])
mention_label_df = pd.DataFrame([(m, 1) for m in pol_mentions] + [(m, 0) for m in nonpol_mentions], columns=['Entity', 'Label'])
print('\n=== Hashtag labels (political=1 then non-political=0) ===')
display(hashtag_label_df.head(10))
display(hashtag_label_df.tail(10))
print('\n=== Mention labels (political=1 then non-political=0) ===')
display(mention_label_df.head(10))
display(mention_label_df.tail(10))

In [ ]:
political_mentions_df['Label'] = ""
political_hashtags_df['Label'] = ""

political_mentions_df.head()
political_hashtags_df.head()

In [ ]:
top_200_mentions = set(political_mentions_df.head(200)['mention'])
top_200_hashtags = set(political_hashtags_df.head(200)['hashtag'])
df['mention_filtered'] = df['mention'].apply(lambda names: [m for m in names if m.lower() in top_200_mentions])
df['hashtags_filtered'] = df['hashtags'].apply(lambda tags: [t for t in tags if t.lower() in top_200_hashtags])
political_mentions = set(mention_political_dict.keys())
political_hashtags = set(hashtag_political_dict.keys())
non_political_mentions = set(mention_non_political_dict.keys())
non_political_hashtags = set(hashtag_non_political_dict.keys())
df['has_political_mention'] = df['mention_filtered'].apply(lambda names: bool(set(names) & political_mentions))
df['has_political_hashtag'] = df['hashtags_filtered'].apply(lambda tags: bool(set(tags) & political_hashtags))
df['has_nonpolitical_mention'] = df['mention_filtered'].apply(lambda names: bool(set(names) & non_political_mentions))
df['has_nonpolitical_hashtag'] = df['hashtags_filtered'].apply(lambda tags: bool(set(tags) & non_political_hashtags))
df['tweet_label'] = (df['has_political_mention'] | df['has_political_hashtag']).astype(int)
total_tweets = len(df)
n_political = df['tweet_label'].sum()
n_nonpolitical = total_tweets - n_political
print(f"Total tweets                                : {total_tweets:,}")
print(f"Tweets labeled political (tweet_label=1)    : {n_political:,} ({n_political/total_tweets*100:.2f}%)")
print(f"Tweets labeled non-political (tweet_label=0): {n_nonpolitical:,} ({n_nonpolitical/total_tweets*100:.2f}%)")
df[['tweet', 'mention', 'hashtags', 'tweet_label']].head(100)


In [ ]:
top_300_mentions = set(political_mentions_df.head(300)['mention'])
top_300_hashtags = set(political_hashtags_df.head(300)['hashtag'])
df['mention_filtered'] = df['mention'].apply(lambda names: [m for m in names if m.lower() in top_200_mentions])
df['hashtags_filtered'] = df['hashtags'].apply(lambda tags: [t for t in tags if t.lower() in top_200_hashtags])
political_mentions = set(mention_political_dict.keys())
political_hashtags = set(hashtag_political_dict.keys())
non_political_mentions = set(mention_non_political_dict.keys())
non_political_hashtags = set(hashtag_non_political_dict.keys())
df['has_political_mention'] = df['mention_filtered'].apply(lambda names: bool(set(names) & political_mentions))
df['has_political_hashtag'] = df['hashtags_filtered'].apply(lambda tags: bool(set(tags) & political_hashtags))
df['has_nonpolitical_mention'] = df['mention_filtered'].apply(lambda names: bool(set(names) & non_political_mentions))
df['has_nonpolitical_hashtag'] = df['hashtags_filtered'].apply(lambda tags: bool(set(tags) & non_political_hashtags))
df['tweet_label'] = (df['has_political_mention'] | df['has_political_hashtag']).astype(int)
total_tweets = len(df)
n_political = df['tweet_label'].sum()
n_nonpolitical = total_tweets - n_political
print(f"Total tweets                                : {total_tweets:,}")
print(f"Tweets labeled political (tweet_label=1)    : {n_political:,} ({n_political/total_tweets*100:.2f}%)")
print(f"Tweets labeled non-political (tweet_label=0): {n_nonpolitical:,} ({n_nonpolitical/total_tweets*100:.2f}%)")
df[['tweet', 'mention', 'hashtags', 'tweet_label']].head(100)


In [ ]:
print(political_mentions_df.columns.tolist())
print(political_hashtags_df.columns.tolist())

In [ ]:
no_mention = df['mention'].apply(lambda x: len(x) == 0)
no_hashtag = df['hashtags'].apply(lambda x: len(x) == 0)
no_both = no_mention & no_hashtag
count_empty = int(no_both.sum())
total = len(df)
print(f" no mention and hashtags= {count_empty:,}/{total:,}")
print(f"% :  {count_empty/total*100:.2f}%")


In [ ]:
print(df.columns.tolist())

In [ ]:
df = df[['tweet','clean_text', 'user', 'mention', 'hashtags', 'tweet_label']]

In [ ]:
df.columns

In [ ]:
type(df.iloc[0].hashtags)

In [ ]:
~df['hashtags'].astype(bool)

In [ ]:
len(df[(~df['mention'].astype(bool))&(~df['hashtags'].astype(bool))])

In [ ]:
df[(~df['mention'].astype(bool))&(~df['hashtags'].astype(bool))]

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_text(text):
    return ' '.join(stemmer.stem(word) for word in str(text).split())

df['stemmed_text'] = df['tweet'].apply(stem_text)

In [ ]:
clean_vocab_list = [w for w, s, d in political_vocab[:700]]

tfidf_pol = TfidfVectorizer(
    vocabulary    = clean_vocab_list,  # restrict to cleaned vocab only
    sublinear_tf  = True
)

X_pol_only    = tfidf_pol.fit_transform(df['clean_text'])
political_cols = list(range(X_pol_only.shape[1]))   # all cols = political cols

political_score = np.asarray(X_pol_only.sum(axis=1)).flatten()
df['political_score'] = political_score

non_zero  = political_score[political_score > 0]
threshold = float(np.median(non_zero))
df['is_political'] = (df['political_score'] > 0).astype(int)     # <-- the 0/1 label assignment

pol_count    = int(df['is_political'].sum())
nonpol_count = int((df['is_political']==0).sum())

print(f"Vocabulary used              : {len(clean_vocab_list)}")
print(f"Identified as POLITICAL      : {pol_count:,}  ({pol_count/len(df)*100:.1f}%)")
print(f"Identified as NON-POLITICAL  : {nonpol_count:,}  ({nonpol_count/len(df)*100:.1f}%)")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer as _TfidfVectorizer

_pol_vocab_list = sorted(set(str(w).lower() for w in political_vocab))
tfidf_vec = _TfidfVectorizer(vocabulary=_pol_vocab_list, lowercase=True, token_pattern=r"(?u)\b[a-z]+\b")
tfidf_matrix = tfidf_vec.fit_transform(df['tweet'].astype(str))

df['tfidf_political_score'] = tfidf_matrix.sum(axis=1).A1.round(5)
df['tfidf_vocab_hits'] = (tfidf_matrix > 0).sum(axis=1).A1.astype(int)
df['tfidf_label'] = (df['tfidf_vocab_hits'] >= 1).astype(int)     # <-- the second 0/1 label assignment

tfidf_political_df = df[df['tfidf_label'] == 1].copy()
tfidf_unlabelled_df = df[df['tfidf_label'] == 0].copy()

print("=== APPROACH A: TF-IDF vocabulary labeling (all tweets) ===")
print(f"Total tweets                       : {len(df):,}")
print(f"Labelled POLITICAL (tfidf_label=1) : {len(tfidf_political_df):,} ({len(tfidf_political_df)/len(df)*100:.2f}%)")
print(f"Labelled NON-POLITICAL (label=0)   : {len(tfidf_unlabelled_df):,} ({len(tfidf_unlabelled_df)/len(df)*100:.2f}%)")
print("tfidf_label value counts:", df['tfidf_label'].value_counts().to_dict())

In [ ]:
# STEP 1: TF-IDF on Full Dataset
# Independent baseline - no labeling yet
# Goal: Apply TF-IDF vectorization to show score distribution

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Apply TF-IDF on full dataset
tfidf = TfidfVectorizer(max_features=1000, min_df=2, max_df=0.8, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Total tweets: {tfidf_matrix.shape[0]}")
print(f"Features (vocabulary size): {tfidf_matrix.shape[1]}")
print(f"Sparsity: {1.0 - (tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))*100:.2f}%")

In [ ]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])  # NER only, faster

RELEVANT_ENTITY_LABELS = {"PERSON", "ORG", "GPE", "NORP"}

# political names collected manually
political_entities = set(
    [x.lower() for x in political_mentions] +
    [x.lower() for x in political_hashtags]
)

# --- restrict to tweets NOT already labelled by mention/hashtag (Stage 1)
# or TF-IDF (Stage 2) — running NER on the full 48k tweets is what hung
# the notebook before ---
no_mention = df['mention'].apply(len) == 0
no_hashtag = df['hashtags'].apply(len) == 0
remaining_mask = no_mention & no_hashtag
if 'tfidf_label' in df.columns:
    remaining_mask = remaining_mask & (df['tfidf_label'] == 0)

ner_idx = df.loc[remaining_mask].index
ner_texts = df.loc[remaining_mask, 'tweet'].astype(str).tolist()
print(f"Tweets entering NER: {len(ner_texts):,} / {len(df):,}")

# --- batched processing via nlp.pipe() instead of per-row nlp() calls ---
entities_list, entity_counts, political_matches_list, labels = [], [], [], []

for doc in nlp.pipe(ner_texts, batch_size=200):
    entities = [ent.text for ent in doc.ents if ent.label_ in RELEVANT_ENTITY_LABELS]
    entity_count = len(entities)
    political_matches = sum(1 for e in entities if e.lower().replace(" ", "") in political_entities)
    label = 1 if political_matches > 0 else 0

    entities_list.append(entities)
    entity_counts.append(entity_count)
    political_matches_list.append(political_matches)
    labels.append(label)

# initialize columns for the full df, then fill only the processed subset
df['ner_entities'] = [[]] * len(df)
df['ner_entity_count'] = 0
df['ner_political_matches'] = 0
df['ner_label'] = 0

df.loc[ner_idx, 'ner_entities'] = pd.Series(entities_list, index=ner_idx, dtype=object)
df.loc[ner_idx, 'ner_entity_count'] = pd.Series(entity_counts, index=ner_idx)
df.loc[ner_idx, 'ner_political_matches'] = pd.Series(political_matches_list, index=ner_idx)
df.loc[ner_idx, 'ner_label'] = pd.Series(labels, index=ner_idx)

In [ ]:
# Create NER labels and display results table

n_political = int(df.loc[ner_idx, 'ner_label'].sum())
n_total = len(ner_idx)

print("=== NER LABELING RESULTS ===")
print(f"Tweets processed by NER            : {n_total:,}")
print(f"Labelled POLITICAL (ner_label=1)   : {n_political:,} ({n_political/n_total*100:.2f}%)")
print(f"Labelled NON-POLITICAL (ner_label=0): {n_total-n_political:,} ({(n_total-n_political)/n_total*100:.2f}%)")

ner_results_table = df.loc[ner_idx, [
    'tweet', 'user', 'mention', 'hashtags',
    'ner_entities', 'ner_entity_count', 'ner_political_matches', 'ner_label'
]]

print("\nSample POLITICAL (ner_label=1):")
display(ner_results_table[ner_results_table['ner_label'] == 1].head(10))

print("\nSample NON-POLITICAL (ner_label=0):")
display(ner_results_table[ner_results_table['ner_label'] == 0].head(10))

ner_results_table.to_csv('ner_labeling_results.csv', index=False)

In [ ]:
# Define political_vocab_set from TF-IDF vocabulary
# Convert the TF-IDF vocabulary (array of words) into a set for fast lookup
political_vocab_set = set(vocab)

In [ ]:
# STEP 3: NMF Topic Modeling on Filtered Dataset
# Unsupervised topic extraction with 5 topics
# User will manually inspect and assign political/non-political labels

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import NMF
import pandas as pd

print("\nSTEP 3: NMF Topic Modeling")
print("=" * 80)

# --- FIX 1: df_remaining must be defined from the tweets still
# unlabelled after mention/hashtag + TF-IDF + NER. Recompute it here
# directly so this cell doesn't silently depend on an earlier cell
# that may not have run (this was the root cause of df_remaining being
# undefined before) ---
no_mention = df['mention'].apply(len) == 0
no_hashtag = df['hashtags'].apply(len) == 0
remaining_mask = no_mention & no_hashtag

if 'tfidf_label' in df.columns:
    remaining_mask = remaining_mask & (df['tfidf_label'] == 0)
if 'ner_label' in df.columns:
    remaining_mask = remaining_mask & (df['ner_label'] == 0)

df_remaining = df.loc[remaining_mask].copy()
print(f"Tweets entering NMF: {len(df_remaining):,}")

# Create CountVectorizer for NMF (better for topic modeling than TF-IDF)
count_vectorizer = CountVectorizer(
    max_features=2000,
    min_df=2,
    max_df=0.8,
    stop_words='english',
    ngram_range=(1, 2)  # unigrams and bigrams
)

# Fit on filtered dataset
count_matrix = count_vectorizer.fit_transform(df_remaining['clean_text'])
print(f"Count matrix shape: {count_matrix.shape}")
print(f"Vocabulary size: {len(count_vectorizer.get_feature_names_out())}")

# Apply NMF with 5 topics
n_topics = 5
print(f"\nTraining NMF model with {n_topics} topics...")
nmf = NMF(n_components=n_topics, random_state=42, max_iter=300, init='nndsvd')
nmf.fit(count_matrix)
print("NMF model trained successfully")

# Display top words for each topic
feature_names = count_vectorizer.get_feature_names_out()

print("\n" + "=" * 80)
print("TOP WORDS FOR EACH TOPIC")
print("=" * 80)

topic_words = {}
for topic_idx, topic in enumerate(nmf.components_):
    top_indices = topic.argsort()[-10:][::-1]  # top 10 words
    top_words = [feature_names[i] for i in top_indices]
    topic_words[topic_idx] = top_words
    print(f"\nTopic {topic_idx}: {', '.join(top_words)}")

# Assign topics to tweets and get topic distribution
W = nmf.transform(count_matrix)  # Document-topic matrix
df_remaining['dominant_topic'] = W.argmax(axis=1)
df_remaining['topic_strength'] = W.max(axis=1)

print("\n" + "=" * 80)
print("TOPIC DISTRIBUTION IN FILTERED DATASET")
print("=" * 80)
print(f"Total tweets analyzed: {len(df_remaining)}")
print("\nTweets per topic:")
print(df_remaining['dominant_topic'].value_counts().sort_index())
print(f"\nAverage topic strength: {df_remaining['topic_strength'].mean():.3f}")
print("\n*** NEXT STEP: Manually inspect topics above and assign political/non-political labels ***")
print("*** Edit topic_label below, then re-run this cell to apply it ***")

# --- FIX 2: removed the stray trailing ')' that caused the SyntaxError
# and made everything above unreachable ---

# fill this in after reading the topic words printed above
# e.g. topic_label = {0: 1, 1: 0, 2: 1, 3: 0, 4: 0}   # 1=political, 0=non-political
topic_label = {}

if topic_label:
    df_remaining['nmf_label'] = df_remaining['dominant_topic'].map(topic_label)

    # --- FIX 3: same safe pd.Series(..., index=...) pattern as the NER fix,
    # to avoid ragged/misaligned assignment back into the full df ---
    df['nmf_label'] = 0
    df.loc[df_remaining.index, 'nmf_label'] = pd.Series(
        df_remaining['nmf_label'].values, index=df_remaining.index
    )

    n_political = int(df.loc[df_remaining.index, 'nmf_label'].sum())
    print(f"\nLabelled POLITICAL (nmf_label=1)   : {n_political:,} ({n_political/len(df_remaining)*100:.2f}%)")
    print(f"Labelled NON-POLITICAL (nmf_label=0): {len(df_remaining)-n_political:,}")
else:
    print("\n⚠️ topic_label is empty — fill it in based on the printed topic words, then re-run this cell.")

In [ ]:
print("\n=== STEP 4: LDA Topic Modeling for Political Tweet Labeling ===")

# Install gensim if needed
import subprocess
subprocess.run(['pip', 'install', '-q', 'gensim'], check=False)

from gensim import corpora, models
from gensim.parsing.preprocessing import STOPWORDS

# Prepare texts for LDA
texts = [tweet.lower().split() for tweet in df_remaining['clean_text']]

# Create dictionary and corpus
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

# Train LDA model with 5 topics (same as NMF for comparison)
print("Training LDA model...")
lda_model = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=5, random_state=42, passes=10, alpha='auto', eta='auto')

print("\nLDA Topics discovered:")
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}: {topic}")

    # Extract dominant LDA topic for each tweet
    df_remaining['lda_dominant_topic'] = [max(lda_model.get_document_topics(bow), key=lambda x: x[1])[0] if lda_model.get_document_topics(bow) else 0 for bow in corpus]

    print("\nLDA Topic distribution:")
    print(df_remaining['lda_dominant_topic'].value_counts().sort_index())

    # LDA-based labeling: Topics containing political keywords get label 1
    political_topics = [1, 4]  # These topics seem related to politics based on keywords
    df_remaining['lda_label'] = df_remaining['lda_dominant_topic'].apply(lambda x: 1 if x in political_topics else 0)

    print(f"\nLDA Labeling Results:")
    print(f"Political tweets (LDA): {(df_remaining['lda_label'] == 1).sum()}")
    print(f"Non-Political tweets (LDA): {(df_remaining['lda_label'] == 0).sum()}")
    print(f"LDA Label distribution:\n{df_remaining['lda_label'].value_counts()}")


=== STEP 4: LDA Topic Modeling for Political Tweet Labeling ===
Training LDA model...

LDA Topics discovered:
Topic 0: 0.006*"coming" + 0.006*"job" + 0.005*"whole" + 0.005*"havent" + 0.005*"says" + 0.005*"whats" + 0.004*"listening" + 0.004*"online" + 0.004*"food" + 0.003*"almost"

LDA Topic distribution:
lda_dominant_topic
0      485
1      486
2      566
3     1081
4    11147
Name: count, dtype: int64

LDA Labeling Results:
Political tweets (LDA): 11633
Non-Political tweets (LDA): 2132
LDA Label distribution:
lda_label
1    11633
0     2132
Name: count, dtype: int64
Topic 1: 0.006*"making" + 0.006*"coffee" + 0.005*"cold" + 0.004*"half" + 0.004*"nite" + 0.004*"ones" + 0.003*"makes" + 0.003*"happened" + 0.003*"bjp" + 0.002*"tea"

LDA Topic distribution:
lda_dominant_topic
0      486
1      486
2      566
3     1080
4    11147
Name: count, dtype: int64

LDA Labeling Results:
Political tweets (LDA): 11633
Non-Political tweets (LDA): 2132
LDA Label distribution:
lda_label
1    11633
0    

In [ ]:
df['political_label'] = 0
df.loc[df['tweet_label'] == 1, 'political_label'] = 1
df.loc[df['tfidf_label'] == 1, 'political_label'] = 1
df.loc[df['ner_label'] == 1, 'political_label'] = 1
df.loc[df['nmf_label'] == 1, 'political_label'] = 1
df.loc[df['lda_label'] == 1, 'political_label'] = 1
print(df['political_label'].value_counts())
print(f"Political: {df['political_label'].mean()*100:.2f}%")

print("\nSample POLITICAL:")
display(df[df['political_label']==1]['tweet'].sample(5, random_state=1))
print("\nSample NON-POLITICAL:")[66]

display(df[df['political_label']==0]['tweet'].sample(5, random_state=1))

df.to_csv('final_political_labels.csv', index=False)

KeyError: 'nmf_label'

In [ ]:
# ============================================================
# STEP: Combine labels using agreement-based voting
# (replaces the plain OR-based df['political_label'] assignment)
# ============================================================

# tweet_label = mention/hashtag family (high-confidence, from your verified top-300 dictionary)
# tfidf_label, ner_label, nmf_label, lda_label = noisier heuristic families

label_cols = ['tfidf_label', 'ner_label', 'nmf_label', 'lda_label']

# make sure any missing family columns default to 0 instead of breaking the sum
for col in label_cols:
    if col not in df.columns:
        df[col] = 0

df['family_agreement_count'] = df[label_cols].sum(axis=1)

MIN_FAMILIES_AGREEING = 2

df['political_label'] = 0

# tweet_label (mention/hashtag) is trusted on its own -> overrides everything
df.loc[df['tweet_label'] == 1, 'political_label'] = 1

# for tweets with NO mention/hashtag signal, require at least 2 of the
# noisier families (tfidf/ner/nmf/lda) to agree before calling it political
no_signal = df['tweet_label'] != 1
df.loc[no_signal & (df['family_agreement_count'] >= MIN_FAMILIES_AGREEING), 'political_label'] = 1

print(df['political_label'].value_counts())
print(f"Political: {df['political_label'].mean()*100:.2f}%")

print("\nSample POLITICAL:")
display(df[df['political_label'] == 1]['tweet'].sample(5, random_state=1))

print("\nSample NON-POLITICAL:")
display(df[df['political_label'] == 0]['tweet'].sample(5, random_state=1))

df.to_csv('final_political_labels.csv', index=False)

In [ ]:
X_text = df['clean_text']
y = df['political_label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

svm_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.9)
X_train = svm_vectorizer.fit_transform(X_train_text)
X_test = svm_vectorizer.transform(X_test_text)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class balance:\n{y_train.value_counts(normalize=True)}")

In [ ]:


from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, confusion_matrix

baseline_model = LinearSVC(max_iter=5000)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)

print(classification_report(y_test, y_pred_baseline, target_names=['Non-Political', 'Political']))

baseline_f1 = f1_score(y_test, y_pred_baseline)
print(f"Baseline Test F1: {baseline_f1:.4f}")

cm_baseline = confusion_matrix(y_test, y_pred_baseline)
print("\nConfusion Matrix (baseline):")
print(cm_baseline)

In [ ]:
# ============================================================
# STEP: SVM (LinearSVC) training with GridSearchCV
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import LinearSVC

# NOTE: only use clean_text/tweet + engineered non-leaky features as input.
# Do NOT include tweet_label, tfidf_label, ner_label, nmf_label, lda_label,
# or family_agreement_count as model features — those columns were used to
# CONSTRUCT political_label, so including them causes label leakage
# (perfect/tautological accuracy that means nothing).

X_text = df['clean_text']
y = df['political_label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

svm_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.9)
X_train = svm_vectorizer.fit_transform(X_train_text)
X_test = svm_vectorizer.transform(X_test_text)

print(f"Train shape: {X_train.shape}")
print(f"Test shape : {X_test.shape}")

param_grid = {'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced']}
grid = GridSearchCV(LinearSVC(max_iter=5000), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best CV F1: {grid.best_score_:.4f}")

svm_model = grid.best_estimator_

In [ ]:
param_grid = {'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced']}
grid = GridSearchCV(LinearSVC(max_iter=5000), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best CV F1: {grid.best_score_:.4f}")

svm_model = grid.best_estimator_

In [ ]:
l_vocab_set


In [ ]:
# ============================================================
# STEP: Tuned SVM — Held-out Test Evaluation
# ============================================================

from sklearn.metrics import classification_report, f1_score, confusion_matrix

y_pred_tuned = svm_model.predict(X_test)

print("=== Tuned SVM (GridSearchCV best params) — Held-out Test Performance ===")
print(classification_report(y_test, y_pred_tuned, target_names=['Non-Political', 'Political']))

tuned_f1 = f1_score(y_test, y_pred_tuned)
print(f"Tuned Test F1: {tuned_f1:.4f}")

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
print("\nConfusion Matrix (tuned):")
print(cm_tuned)

print(f"\nBaseline Test F1: {baseline_f1:.4f}")
print(f"Tuned Test F1   : {tuned_f1:.4f}")
print(f"Improvement     : {tuned_f1 - baseline_f1:+.4f}")

In [ ]:
# ============================================================
# STEP: Error analysis — inspect misclassified tweets
# ============================================================

test_results = pd.DataFrame({
    'tweet': X_test_text.values,
    'true_label': y_test.values,
    'predicted_label': y_pred_tuned
})

false_positives = test_results[(test_results['true_label'] == 0) & (test_results['predicted_label'] == 1)]
false_negatives = test_results[(test_results['true_label'] == 1) & (test_results['predicted_label'] == 0)]

print(f"False positives (predicted political, actually non-political): {len(false_positives)}")
print(f"False negatives (predicted non-political, actually political): {len(false_negatives)}")

print("\nSample False Positives:")
display(false_positives['tweet'].sample(min(5, len(false_positives)), random_state=1))

print("\nSample False Negatives:")
display(false_negatives['tweet'].sample(min(5, len(false_negatives)), random_state=1))

In [ ]:
# ============================================================
# STEP: Persist the trained model + vectorizer
# ============================================================

import joblib

joblib.dump(svm_model, 'political_classifier_svm.joblib')
joblib.dump(svm_vectorizer, 'political_tfidf_vectorizer.joblib')

print("Saved: political_classifier_svm.joblib")
print("Saved: political_tfidf_vectorizer.joblib")

In [ ]:
loaded_model = joblib.load('political_classifier_svm.joblib')
loaded_vectorizer = joblib.load('political_tfidf_vectorizer.joblib')

In [ ]:
# ============================================================
# STEP: Test the trained model on brand-new, unseen tweets
# ============================================================

new_tweets = [
    "@narendramodi bjp government announces new policy for farmers welfare",
    "just watched a great movie with my friends tonight, loved it",
    "rahulgandhi congress slams government over inflation and unemployment",
    "happy birthday to my sister, wishing you all the best",
    "@amitshah addresses rally ahead of state elections",
    "@viratkohli scores century in today's match against australia",
    "#farmersprotest continues as farmer unions reject new bills",
    "excited to try out the new restaurant that opened near my house"
]

# IMPORTANT: use .transform(), never .fit_transform(), on new data —
# it must reuse the exact vocabulary/weights learned during training
X_new = svm_vectorizer.transform(new_tweets)

predictions = svm_model.predict(X_new)

results_df = pd.DataFrame({
    'tweet': new_tweets,
    'predicted_label': predictions,
    'prediction': ['Political' if p == 1 else 'Non-Political' for p in predictions]
})

display(results_df)


In [117]:
# ============================================================
# STEP: Final sample-tweet inference table
# tweet | mentions | hashtags | NER | NMF | LDA | predicted_label | predicted
# Each of mentions/hashtags/NER/NMF/LDA shows the actual matched
# words/topic extracted from that tweet — the label is derived from
# those extracted features (not the SVM's own TF-IDF vectorizer).
# ============================================================

import re
import numpy as np
import pandas as pd

new_tweets = [
    "@narendramodi bjp government announces new policy for farmers welfare",
    "just watched a great movie with my friends tonight, loved it",
    "rahulgandhi congress slams government over inflation and unemployment",
    "happy birthday to my sister, wishing you all the best",
    "@amitshah addresses rally ahead of state elections",
    "@viratkohli is the greatest cricketer in my opinion",
    "#BJP continues their crulety as farmer unions reject new bills",
    "excited to try out the new restaurant that opened near my house"
]

political_mentions_set = set(mention_political_dict.keys())
political_hashtags_set = set(hashtag_political_dict.keys())

rows = []

for tweet in new_tweets:
    clean = preprocess(tweet)

    # --- mentions actually found in the tweet ---
    mentions_found = [m.lower() for m in re.findall(r'@(\w+)', tweet)]

    # --- hashtags actually found in the tweet ---
    hashtags_found = [h.lower() for h in re.findall(r'#(\w+)', tweet)]

    # --- NER entities extracted from the tweet ---
    doc = nlp(tweet)
    ner_entities = [ent.text for ent in doc.ents if ent.label_ in RELEVANT_ENTITY_LABELS]

    # --- NMF: top word(s) driving the assigned topic for this tweet ---
    nmf_words = []
    try:
        vec = count_vectorizer.transform([clean])
        topic_dist = nmf.transform(vec)[0]
        top_topic = int(np.argmax(topic_dist))
        feat_names = count_vectorizer.get_feature_names_out()
        top_term_idx = nmf.components_[top_topic].argsort()[::-1][:5]
        candidate_terms = [feat_names[i] for i in top_term_idx]
        # only keep terms that actually appear in this tweet's clean text
        nmf_words = [t for t in candidate_terms if t in clean]
    except Exception:
        pass

    # --- LDA: top word(s) driving the assigned topic for this tweet ---
    lda_words = []
    try:
        bow = dictionary.doc2bow(clean.split())
        topic_probs = lda_model.get_document_topics(bow)
        if topic_probs:
            top_topic = max(topic_probs, key=lambda x: x[1])[0]
            candidate_terms = [w for w, _ in lda_model.show_topic(top_topic, topn=10)]
            lda_words = [t for t in candidate_terms if t in clean]
    except Exception:
        pass

    # --- derive label from the extracted features (agreement vote) ---
    has_political_mention = bool(set(mentions_found) & political_mentions_set)
    has_political_hashtag = bool(set(hashtags_found) & political_hashtags_set)
    tweet_label = int(has_political_mention or has_political_hashtag)

    ner_label  = int(any(e.lower().replace(" ", "") in (political_mentions_set | political_hashtags_set) for e in ner_entities))
    nmf_label  = int(len(nmf_words) > 0)
    lda_label  = int(len(lda_words) > 0)

    family_agreement_count = ner_label + nmf_label + lda_label
    predicted_label = 1 if tweet_label == 1 else int(family_agreement_count >= 2)

    rows.append({
        'tweet': tweet,
        'mentions': mentions_found,
        'hashtags': hashtags_found,
        'NER': ner_entities,
        'NMF': nmf_words,
        'LDA': lda_words,
        'predicted_label': predicted_label,
        'predicted': 'Political' if predicted_label == 1 else 'Non-Political'
    })

final_inference_table = pd.DataFrame(rows)
display(final_inference_table)

,tweet,mentions,hashtags,NER,NMF,LDA,predicted_label,predicted
0,@narendramodi bjp government announces new policy for farmers welfare,[narendramodi],[],[],[bjp],"[modi, bjp]",1,Political
1,"just watched a great movie with my friends tonight, loved it",[],[],[],[],[love],0,Non-Political
2,rahulgandhi congress slams government over inflation and unemployment,[],[],[congress],[congress],[],1,Political
3,"happy birthday to my sister, wishing you all the best",[],[],[],"[day, happy]",[day],1,Political
4,@amitshah addresses rally ahead of state elections,[amitshah],[],[],[],[],1,Political
5,@viratkohli is the greatest cricketer in my opinion,[viratkohli],[],[],[],[],0,Non-Political
6,#BJP continues their crulety as farmer unions reject new bills,[],[bjp],[BJP],[bjp],[],1,Political
7,excited to try out the new restaurant that opened near my house,[],[],[],[new],[],0,Non-Political
